In [541]:
import pdfplumber
import pandas as pd
import os
import re
from img2table.document import Image
from io import BytesIO
from pdf2image import convert_from_path
from img2table.ocr import TesseractOCR
from img2table.document import PDF
import json

In [542]:
ocr = TesseractOCR()

tesseract 5.5.2
 leptonica-1.87.0
  libgif 5.2.2 : libjpeg 8d (libjpeg-turbo 3.1.3) : libpng 1.6.58 : libtiff 4.7.2 : zlib 1.2.12 : libwebp 1.6.0 : libopenjp2 2.5.4
 Found NEON
 Found libarchive 3.8.8 zlib/1.2.12 liblzma/5.8.3 bz2lib/1.0.8 liblz4/1.10.0 libzstd/1.5.7 expat/expat_2.7.3 CommonCrypto/system libb2/system
 Found libcurl/8.7.1 SecureTransport (LibreSSL/3.3.6) zlib/1.2.12 nghttp2/1.64.0


In [543]:
def extract_headers(page):
    lines = page.extract_text_lines()
    return pd.DataFrame([line for line in lines if re.search('^[A-Z]{1}[0-9]{1,2}', line['text']) is not None])

In [544]:
def select_best_heading(page, table, headers):
    if len(headers) == 0:
        return ''
    headers['is_above'] = headers['top'].apply(lambda x: x < table.bbox.relative.y1 * page.height)
    headers = headers[headers['is_above']].sort_values('top', ascending=False).reset_index(drop=True)
    if len(headers) == 0:
        return ''
    return headers['text'][0]

In [545]:
def parse_page_tables(file, page_index):
    pdf = pdfplumber.open(file)
    page = pdf.pages[page_index]
    headers = extract_headers(page)

    doc = PDF(file,
          pages=[page_index],
          detect_rotation=False,
          pdf_text_extraction=True)

    results = doc.extract_tables(ocr=ocr,
                    implicit_rows=False,
                    implicit_columns=False,
                    borderless_tables=False,
                    min_confidence=50,
                    max_workers=1)

    data = []
    for table in results[page_index]:
        heading = select_best_heading(page, table, headers)
        code_match = re.search('^[A-Z]{1}[0-9]+', heading)
        if code_match is not None:
            code = code_match.group(0).strip()
        else:
            code = ''

        data.append({
            'table_code': code,
            'table_title': heading.replace(code, '').strip(),
            'table_records': table.df.fillna('').to_dict(orient='records'),
            'table_html': re.sub('[\n ]+', ' ', table.html)
        })

    return data

In [546]:
def first(lst):
    return list(lst)[0]

In [547]:
years = os.listdir('../data/cds-docs')
year = '2024-2025'

In [548]:
files = [f'../data/cds-docs/{year}/{file}' for file in os.listdir(f'../data/cds-docs/{year}')]

In [549]:
file = files[5]

In [550]:
unitid = file.split('/')[-1].replace('.pdf', '').strip()

In [551]:
data = []
for page in range(0, len(pdf.pages)):
    data.extend(parse_page_tables(file, page))

In [552]:
data = pd.DataFrame(data)

In [553]:
data = data[data['table_code'] != '']

In [554]:
data = data.groupby('table_code').agg({
    'table_code': first,
    'table_title': first,
    'table_records': list,
    'table_html': lambda tables: '<div class="separator"></div>'.join(tables)
})

In [555]:
data = {
    unitid: data.to_dict(orient='records')
}

In [556]:
# os.system(f'open {file}')

In [557]:
with open('../web-app/data.json', 'w') as out_file:
    json.dump(data, out_file)

In [558]:
# with open('test.html', 'w') as out_file:
#     out_file.write(results[4][0].html)